## PCA loadings

Visualizes `cache/pca_loadings.csv` — for each zero-shot label, its loading on the first 5 PCA components (see `experiments/SentencePca.kt`). One horizontal bar chart per component, bars sorted by loading value, colored on a diverging scale centered at zero.

For each component, also visualizes the most/least representative sentences from `cache/sentence_features.csv` — the sentences whose projection onto that component is highest and lowest.

In [1]:
%useLatestDescriptors
%use dataframe
%use kandy

In [2]:
import kotlin.math.abs

val loadings = DataFrame.readCsv("../data/sentence_pca_loadings.csv")
val pcColumns = loadings.columnNames().filter { it.startsWith("pc") }

val pcValues: List<List<Double>> = pcColumns.map { col ->
    (loadings[col] as DataColumn<*>).toList().map { (it as Number).toDouble() }
}

val maxAbsLoading = ArrayList<Double>(loadings.rowsCount())
for (i in 0 until loadings.rowsCount()) {
    var maxAbs = 0.0
    for (col in pcValues) {
        val v = abs(col[i])
        if (v > maxAbs) maxAbs = v
    }
    maxAbsLoading.add(maxAbs)
}

val loadingsWithMax = (loadings + maxAbsLoading.toColumn("maxAbsLoading")).sortByDesc("maxAbsLoading")

println("Rows: ${loadingsWithMax.rowsCount()}, columns: ${loadingsWithMax.columnNames()}")
loadingsWithMax

kotlin-logging: initializing... active logger factory: Slf4jLoggerFactory
Rows: 86, columns: [label, pc1, pc2, pc3, pc4, pc5, maxAbsLoading]


label,pc1,pc2,pc3,pc4,pc5,maxAbsLoading
neutral,-0.051773,-0.119543,0.121314,-0.047579,0.439001,0.439001
self-reflection,-0.238847,0.331498,-0.067257,0.011523,0.064795,0.331498
conceptual,-0.056091,0.014520,0.238697,0.329682,0.054004,0.329682
vague,-0.081688,0.007177,0.329094,-0.091604,0.032293,0.329094
self-perception,-0.190607,0.316126,-0.041214,0.016854,0.044285,0.316126
specific,0.094628,-0.013364,-0.310464,-0.026712,0.036958,0.310464
low concentration,-0.089075,0.032825,0.176240,-0.309721,-0.163608,0.309721
positive,0.303097,0.240947,0.062863,0.072257,-0.125255,0.303097
non-conceptual,0.017032,0.005134,0.161714,-0.297947,-0.026523,0.297947
negative,-0.267080,-0.070838,-0.136225,-0.079882,-0.288584,0.288584


In [3]:
fun plotLoadings(component: String) = run {
    val sorted = loadings.sortBy(component)
    val labels = sorted["label"].toList().map { it.toString() }
    val values = sorted[component].toList().map { (it as Number).toDouble() }

    plot {
        layout {
            title = "Label weights in PCA component \"$component\""
            size = 700 to 1600
        }
        barsH {
            y(labels) { axis.name = "label" }
            x(values) { axis.name = "loading" }
            fillColor(values) {
                scale = continuousColorGradient2(Color.BLUE, Color.WHITE, Color.RED, 0.0)
                legend.name = "loading"
            }
        }
    }
}

In [4]:
val features = DataFrame.readCsv("../data/sentence_features.csv").distinctBy("sentence")
println("Rows: ${features.rowsCount()}, columns: ${features.columnNames()}")
features.head()

Rows: 33428, columns: [msgId, date, sentence, pc1, pc2, pc3, pc4, pc5]


msgId,date,sentence,pc1,pc2,pc3,pc4,pc5
25603102,2023-07-23T11:26:08,Previous logs in chronological order:...,0.731455,-1.005547,-0.920180,-0.065204,0.475963
25603102,2023-07-23T11:26:08,"July 25th, 2023 For this log I’m taki...",0.611695,-1.096611,-0.858788,0.209241,0.589878
25603102,2023-07-23T11:26:08,My pawo is staying with me here in Sw...,1.303939,-0.555023,-0.655493,-0.175434,0.161072
25603102,2023-07-23T11:26:08,In the tantra class we learn basic te...,0.926311,-0.823070,-0.390654,0.094769,0.485873
25603102,2023-07-23T11:26:08,We have tried out some different mant...,0.615658,-0.575613,-0.454940,-0.036820,0.512182


In [5]:
fun truncate(s: String, maxLength: Int = 90) =
    if (s.length <= maxLength) s else s.take(maxLength - 1) + "…"

fun printSentences(component: String, n: Int = 5) = run {
    val sorted = features.sortByDesc(component)
    val top = sorted.take(n)
    val bottom = sorted.takeLast(n)

    fun printRows(rows: AnyFrame) {
        val sentences = rows["sentence"].toList().map { truncate(it.toString()) }
        val values = rows[component].toList().map { (it as Number).toDouble() }
        sentences.zip(values).forEach { (sentence, value) ->
            println("%+.3f  %s".format(value, sentence))
        }
    }

    printRows(top)
    println("...")
    printRows(bottom)
}

### Component 1

In [6]:
printSentences("pc1")
plotLoadings("pc1")

+2.551  I'm happy to see that this forum is getting really active, with lots of engaged posts and…
+2.525  Marvellous!
+2.417  Oh yay!
+2.412  Loved the clarity.
+2.398  Great!
...
-2.678  Wouldn’t it be more logical to just dissolve this ”I” who cannot even see the things beca…
-2.687  My body is incompatible with itself.
-2.714  My identity is falling through the cracks.
-2.731  Maybe that was just me being lazy, I don't know, but I think I need to be more flexible b…
-2.733  My mind was apparently not unified.


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="GciW8i" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 700.0, 
 height: 1600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("GciW8i");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Label weights in PCA component \"pc1\""
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":700.0,
"height":1600.0
},
"kind":"plot",
"scales":[{
"aesthetic":"y",
"discrete":true,
"name":"label"
},{
"aesthetic":"x",
"name":"loading",
"limits":[null,null]
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_gradient2",
"high":"#ee6666",
"low":"#5470c6",
"name":"loading",
"mid":"#ffffff",
"midpoint":0.0,
"limits":[null,null]
}],
"layers":[{
"mapping":{
"y":"y",
"x":"x",
"fill":"fill"
},
"stat":"identity",
"orientation":"y",
"data":{
"x":[-0.26707999512861214,-0.24191545382568294,-0.23884696093520008,-0.201549761398695,-0.1990964830684758,-0.19887222345700234,-0.19157941722267272,-0.19060695671519037,-0.1795622580560282,-0.16691566658915336,-0.15282839615764268,-0.13402287398823018,-0.13363366328802337,-0.11234470799442821,-0.1088030812928911,-0.10760705911473019,-0.10626254462793755,-0.10593783724007415,-0.10158098373167795,-0.09644207145626658,-0.09470001316472369,-0.09160087718172828,-0.08907481285778479,-0.08653921008937084,-0.08168768208016342,-0.07609399163189157,-0.07521488840970864,-0.07172666988042348,-0.06125449273590294,-0.06082772509818083,-0.05941652959631235,-0.056411691189810036,-0.05609051439587939,-0.055774420024212845,-0.054880204898669904,-0.051772557643191275,-0.05173518386462849,-0.05129365742355357,-0.045535463058847855,-0.043470810861335134,-0.0351029609383326,-0.03453762264905305,-0.02949958727860151,-0.028823845145896428,-0.027488969421398474,-0.026405938035444852,-0.025110791322072733,-0.017630919624263396,-0.01648093807264673,-0.014639724150147098,-0.013245966274319261,-0.011332649460907215,-0.01097234856694771,2.0125121784394305E-4,0.004077149313015308,0.007489615508378664,0.007590641409487328,0.009468903846512637,0.017031855483211546,0.018075373659764925,0.021668108008887133,0.030669389781846756,0.03172592399438057,0.03769019224490434,0.038713239798971504,0.04189096841436676,0.04196161447295118,0.04210839514383721,0.04780417088807698,0.05630079182740326,0.06534633820873197,0.07246168064052987,0.08090540829751765,0.08512333299299712,0.0946284790808482,0.10272270891651325,0.10639411939114814,0.10928418040359353,0.11287513922995551,0.12569438028845883,0.14230068301552842,0.1463787341386408,0.15219149955776565,0.1613834917337802,0.1721826905202239,0.3030971766362289],
"y":["negative","struggling","self-reflection","self-criticism","uncertainty","self-awareness","dissonance","self-perception","lack of agency","mental","self-reference","author referring to themselves","equanimity wrt negative","confusion","dualities","depersonalization","self-referential","self-judgment","unable to do sth.","weak sense of self","opposites","derealization","low concentration","apprehensive","vague","somatic","passivity","being challenged","weird","polarities","pain","abstract","conceptual","metaphorical","bad mood","neutral","subjective","forced to react","unable to react","paradox","body","past","dullness","awareness","no choice","mystical experience","direct experience","attention","mental images"

### Component 2

In [7]:
printSentences("pc2")
plotLoadings("pc2")

+3.135  I felt high, and my body posture was great all on its own.
+3.067  I can feel them flow through my body, and there are pleasant tingles and spaciousness and…
+2.961  Now after the evening practice, I feel vibrantly alive and yet tranquil.
+2.949  I closed my inner door, smiled, and shivered with happiness.
+2.907  It felt great, resting in awareness like this.
...
-1.670  Unsure about the order here.
-1.685  I have no idea what you are talking about here.
-1.690  We use words very differently, and since words are all we've got here, I have no way of d…
-1.699  I don’t know anything about their attainments.
-1.716  I don’t know what you are referring to.


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="RIrE8H" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 700.0, 
 height: 1600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("RIrE8H");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Label weights in PCA component \"pc2\""
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":700.0,
"height":1600.0
},
"kind":"plot",
"scales":[{
"aesthetic":"y",
"discrete":true,
"name":"label"
},{
"aesthetic":"x",
"name":"loading",
"limits":[null,null]
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_gradient2",
"high":"#ee6666",
"low":"#5470c6",
"name":"loading",
"mid":"#ffffff",
"midpoint":0.0,
"limits":[null,null]
}],
"layers":[{
"mapping":{
"y":"y",
"x":"x",
"fill":"fill"
},
"stat":"identity",
"orientation":"y",
"data":{
"x":[-0.11954307904988548,-0.09674515236140685,-0.088183137513146,-0.07083758534588602,-0.057452685512049156,-0.05233376775821665,-0.049128919578537135,-0.043124641854987496,-0.038127831348264064,-0.03716746800580099,-0.03593784169046545,-0.03061628213473192,-0.02984178213478355,-0.025065791846754486,-0.024824711787647745,-0.01946513068642419,-0.014933383988243766,-0.014033274620729508,-0.013363810662648563,-0.01206046171644382,-0.011515697557841898,-0.010970301483322363,-0.0038801665210532123,-0.0021057491278391183,-6.535888109801592E-4,-1.0625511516128237E-4,0.005134074126178872,0.0053685081469558036,0.007176726208951824,0.0107964874732345,0.014520170085457683,0.015745360626806166,0.020040412064828754,0.021017658326976323,0.021640705997735092,0.025596324830464628,0.026240581343082273,0.02690247089708185,0.029686466562234996,0.03180220667172494,0.03282527607297325,0.03456606868996281,0.04147408102794942,0.042241435988674525,0.043256464687301176,0.04707238378021504,0.05382692163668565,0.05640232387188445,0.05652372306976121,0.05854819069877677,0.05935830112867833,0.06404914954297922,0.06516606419531323,0.0686666524375208,0.07445963238445819,0.07805205781527408,0.0863432492211948,0.0877409745956417,0.09178888427693307,0.0978950178553639,0.09793189407650413,0.09893042720526479,0.09977218054370411,0.10376076144853037,0.1073034021534251,0.11154360362461127,0.11475055713997287,0.12120066713378894,0.12891536513501625,0.1346473012017052,0.14086778173597903,0.15006507001438643,0.15604351265772493,0.15997065723683995,0.1634926655444784,0.1669737593683194,0.16700592411238566,0.16728834120522462,0.18343270763041486,0.18521710925782525,0.22059794526952303,0.24094650379450167,0.245400111487881,0.2838519099225065,0.31612634462229494,0.33149844074149776],
"y":["neutral","uncertainty","objective","negative","dissonance","confusion","auditory","lack of agency","being challenged","struggling","visual","unable to do sth.","author referring to someone else","abstract","apprehensive","no choice","forced to react","depersonalization","specific","measurable","paradox","unable to react","pain","bad mood","weird","turtles","non-conceptual","dullness","vague","derealization","conceptual","mental images","subjective","easily able to do sth.","phenomena that change over seconds","weak sense of self","sounds","self-judgment","equanimity wrt opposites","opposites","low concentration","polarities","high concentration","self-criticism","mystical experience","equanimity wrt negative","dualities","alertness"

### Component 3

In [8]:
printSentences("pc3")
plotLoadings("pc3")

+2.595  It doesn't matter what is up and what is down.
+2.518  Just being with the flow and sensing everything vanish on different scales and in differe…
+2.481  Kind of a softening, a gentle breeze, a light...?
+2.392  Whatever happens is the way of the universe.
+2.364  It's fine just as it is - sort of a chrystal clear confusion.
...
-1.864  During the work meetings, I payed attention to my reactive behavior and how it caused suf…
-1.869  I probably squeeze the nerve higher up when lying in shavasana, because that's when the p…
-1.922  I recognized postures that I know were very painful just days ago, and now by body sinks …
-1.926  I very palpably learned a lesson about suffering tonight as I hurt my muscles trying to c…
-1.980  My body hurts in a way that tells me that I definitely had a reaction to the food (uhm, w…


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="lAJJux" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 700.0, 
 height: 1600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("lAJJux");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Label weights in PCA component \"pc3\""
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":700.0,
"height":1600.0
},
"kind":"plot",
"scales":[{
"aesthetic":"y",
"discrete":true,
"name":"label"
},{
"aesthetic":"x",
"name":"loading",
"limits":[null,null]
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_gradient2",
"high":"#ee6666",
"low":"#5470c6",
"name":"loading",
"mid":"#ffffff",
"midpoint":0.0,
"limits":[null,null]
}],
"layers":[{
"mapping":{
"y":"y",
"x":"x",
"fill":"fill"
},
"stat":"identity",
"orientation":"y",
"data":{
"x":[-0.3104636903612985,-0.16973235030701417,-0.15137835994392596,-0.13726467361012729,-0.13622541304698413,-0.1334718166674683,-0.10864962220379525,-0.09905040392123125,-0.09549839512920145,-0.09092074905322449,-0.07290288473776703,-0.07157444745470071,-0.07036195305972662,-0.06725667241346148,-0.06361122924640734,-0.062117445363481544,-0.06211063923162841,-0.04886293014281804,-0.04301528524422771,-0.04257111882180618,-0.04121440285593208,-0.040550004429630825,-0.03796827100008004,-0.021506761457996904,-0.020660626974027884,-0.016373554190596242,-0.011206785666385237,-0.010937337504283797,-0.008445310150950795,-0.004710322165711746,-0.003812082934976446,9.856183954194437E-5,0.003516921768351627,0.008006642220091181,0.008508565240355382,0.009475338874770882,0.015470446283608802,0.01634758144406407,0.022053500517761865,0.02408577300445259,0.027721373636487214,0.02886879068694463,0.02927586167624093,0.02955676460894313,0.03036813603733146,0.03337423595968915,0.03363019432652247,0.03752786804491121,0.03794240752765815,0.04254023641980929,0.042568790771789045,0.04501039561175093,0.045571760057311755,0.04682716665887478,0.05019373371604387,0.05363573194158473,0.06286265238808551,0.06439065382175675,0.06562953950722197,0.06750378185533917,0.07259831500304686,0.0792812253865663,0.08511549628983356,0.09668829678166892,0.09693059401831472,0.0978308714235222,0.09950366425040004,0.10517125114516326,0.11357996214932076,0.117462717874727,0.12095171568672172,0.12131430480208702,0.14127174806450965,0.1487211838741824,0.15033875669084545,0.16171376871638743,0.16288064513801429,0.16480001931382693,0.16784761735830922,0.17624035298313562,0.19988147711546622,0.20141340167851446,0.20360472620582526,0.23471271602799013,0.23869651086157312,0.3290936838325518],
"y":["specific","direct experience","self-reference","self-referential","negative","able to react","self-criticism","sense of agency","author referring to themselves","body","pain","measurable","capable of doing sth.","self-reflection","self-awareness","self-judgment","somatic","forced to react","past","high concentration","self-perception","confident","bad mood","author referring to someone else","struggling","alertness","attention","being challenged","focus","concentration","energetic","turtles","phenomena that change over seconds","no choice","dualities","weak sense of self","dissonance","apprehensive","polarities","easily able to do sth.","hopeful","excited","visual","auditory","opposites","impressed","awareness","dullness","paradox","

### Component 4

In [9]:
printSentences("pc4")
plotLoadings("pc4")

+1.927  If they really shake up the taken-for-grantedness of human perception, that's awesome, an…
+1.799  I can see the awakened quality of both the fear and the judging of the fear.
+1.783  I find it fascinating how fast this makes dualities collapse for me.
+1.748  I think this is the way to entering lucid dreaming from meditation while being awake in t…
+1.706  So for me, tuning into intent is almost like archeology.
...
-2.116  So I sat for 45 minutes, and sitting was easy.
-2.118  Sitting was comfortable and still.
-2.137  Fell asleep.
-2.161  Afternoon: went to a forest lake.
-2.170  Peaceful, nothing special.


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="UMDfZ2" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 700.0, 
 height: 1600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("UMDfZ2");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Label weights in PCA component \"pc4\""
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":700.0,
"height":1600.0
},
"kind":"plot",
"scales":[{
"aesthetic":"y",
"discrete":true,
"name":"label"
},{
"aesthetic":"x",
"name":"loading",
"limits":[null,null]
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_gradient2",
"high":"#ee6666",
"low":"#5470c6",
"name":"loading",
"mid":"#ffffff",
"midpoint":0.0,
"limits":[null,null]
}],
"layers":[{
"mapping":{
"y":"y",
"x":"x",
"fill":"fill"
},
"stat":"identity",
"orientation":"y",
"data":{
"x":[-0.30972067362033057,-0.29794657486302767,-0.23691635938932318,-0.22176796996358186,-0.20575903169804688,-0.13691292446987788,-0.13560637490862967,-0.13146843752454287,-0.12485939452846184,-0.12060673316211612,-0.10476734778005059,-0.09724822494109836,-0.09578924105316221,-0.09399586195876462,-0.09160446835807781,-0.08237063361860895,-0.07988226374828476,-0.07452019176343337,-0.074100457671855,-0.07023674600324253,-0.06490905795347424,-0.0633009756855708,-0.05690166972156516,-0.056493057949030354,-0.056132565588342336,-0.05205035776256622,-0.04757855052599336,-0.044545425430731825,-0.041919697911678926,-0.03451878542620674,-0.02992000508596139,-0.028682449123574413,-0.02818053774562778,-0.026711565744984588,-0.022877322695688453,-0.02161998050411233,-0.02126582711001259,-0.017043288887622757,-0.016635415532750275,-0.01257825430083312,-0.011712624709124425,-0.009017579579296012,-0.007917626909685506,-0.006152737895973814,-2.458244507618931E-4,0.0014500722596912732,0.00254470806598943,0.007800955263437855,0.009442433157839854,0.010450088445410777,0.011523104618240045,0.01567009829733383,0.01685417150527475,0.02445406399458716,0.03480751790081887,0.039678740881848196,0.04272758714775382,0.04297002073439969,0.044810388781125934,0.046185097771468245,0.048704963249602946,0.05416072244860783,0.05545813129730491,0.05647023388918042,0.06043637372203104,0.0617190948719443,0.06564263736969365,0.06982689373981261,0.06997403321456303,0.0722571371141193,0.07656106567983041,0.0781626610663512,0.07954284757225125,0.08068473584873939,0.08236484359327044,0.08327358379183111,0.09662379268994178,0.09901299326245953,0.11095664412617173,0.12312227747888617,0.13343882433355397,0.19368694251486165,0.22468173319382967,0.24674423290808162,0.2504547076595259,0.32968230778741636],
"y":["low concentration","non-conceptual","lack of agency","passivity","direct experience","dullness","past","calmness","things happening on their own","auditory","objective","depersonalization","unable to do sth.","unable to react","vague","somatic","negative","equanimity","feeling content","body","sensory","easily able to do sth.","no choice","struggling","equanimity wrt negative","satisfaction","neutral","measurable","happiness","sounds","joyful","forced to react","weak sense of self","specific","pain","harmony","self-acceptance","self-reference","good mood","bad mood","self-criticism","author referring to themselves","phenomena that change over seconds","derealization","turtles","visual","sense of agency","confident"

### Component 5

In [10]:
printSentences("pc5")
plotLoadings("pc5")

+2.079  I sat for about 90 minutes and was mindful all the time.
+2.012  I sat for 30 minutes, focusing widely on the breath.
+1.999  Today I have meditated for about four hours so far.
+1.936  I have been meditating for close to four hours today so far.
+1.926  I have tried to be mindful and stay focused.
...
-1.815  I feel as if there are champagne bubbles trickling out through my ears and through the po…
-1.826  And Holy Crap!
-1.892  Bah, more rapture.
-1.962  I’m all bubbly now and the nada sound rings in my ears.
-1.976  I feel like my heart is cracking open.


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="Fdmjka" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 700.0, 
 height: 1600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("Fdmjka");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Label weights in PCA component \"pc5\""
},
"mapping":{
},
"data":{
},
"ggsize":{
"width":700.0,
"height":1600.0
},
"kind":"plot",
"scales":[{
"aesthetic":"y",
"discrete":true,
"name":"label"
},{
"aesthetic":"x",
"name":"loading",
"limits":[null,null]
},{
"aesthetic":"fill",
"scale_mapper_kind":"color_gradient2",
"high":"#ee6666",
"low":"#5470c6",
"name":"loading",
"mid":"#ffffff",
"midpoint":0.0,
"limits":[null,null]
}],
"layers":[{
"mapping":{
"y":"y",
"x":"x",
"fill":"fill"
},
"stat":"identity",
"orientation":"y",
"data":{
"x":[-0.2885837670044262,-0.18293559539491727,-0.17163832383547872,-0.1679088575806657,-0.1673489366592057,-0.166315239175438,-0.16360797699928117,-0.15504615314746667,-0.13943202192483836,-0.1390786533841373,-0.1296176740932151,-0.12567753547816551,-0.12525452718024505,-0.12228498735707642,-0.12019382185237855,-0.11195190278968128,-0.10456768171977161,-0.10429353254285803,-0.09429175303762531,-0.0851835998132808,-0.084278826104485,-0.08250962873942552,-0.07201122072011733,-0.06332297147656624,-0.0629696927299982,-0.061974283719478954,-0.06139211851123535,-0.05947052361601136,-0.057535288072600846,-0.05139051028004023,-0.049770663376742054,-0.04951106286304244,-0.046475941029562116,-0.044751827415117476,-0.03978841671430537,-0.038997634770443386,-0.03731621774275526,-0.03724545470703752,-0.032185252449939326,-0.03082738231056747,-0.028797716484962824,-0.028051910618810367,-0.026523463298456666,-0.025752147861497623,-0.024097180126646994,-0.024034419424690496,-0.01832926771346234,-0.012373807166261491,-0.008676779834548426,-0.00844605790524064,-0.006476896650304729,-0.006368125835681138,-3.715893275149162E-4,0.021001443579684025,0.024565044048740554,0.02602888681749064,0.02674139245854268,0.02681508266839172,0.026881512097939136,0.029259099487313707,0.02978732611829206,0.0322928744706298,0.036957899577722164,0.044285112264159175,0.046698552869738834,0.05384192995452371,0.05400353736855282,0.055046302832138364,0.05654622786614911,0.06030112323917865,0.06479506285053147,0.06931669800734987,0.07852643486889707,0.08617747146297314,0.09676892939298169,0.09834490846022399,0.12667522545766052,0.12904780568485863,0.13841064796102637,0.15235560574458556,0.15440512514026855,0.1683772130361769,0.1754445901946213,0.1769715194212175,0.18603842137508225,0.43900070160667],
"y":["negative","struggling","joyful","dissonance","good mood","impressed","low concentration","excited","happiness","able to react","subjective","lack of agency","positive","sensory","weird","choice","energetic","satisfaction","unable to do sth.","pain","bad mood","forced to react","confusion","auditory","weak sense of self","metaphorical","opposites","self-criticism","attention","dualities","feeling content","being challenged","polarities","derealization","apprehensive","self-judgment","phenomena that change over seconds","mental images","mystical experience","author referring to someone else","visual","hopeful","non-conceptual","paradox","sounds","no choice","body","uncertainty","direct experience","depersonalization",